# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source

The dataset is described using a Croissant schema and is available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

We start by loading the dataset metadata and exploring its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset and display its metadata summary
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")


## 2. Data Overview

Let's list all available record sets (tables), their `@id` fields, and the fields within each. We'll use `mlcroissant`'s Python API to enumerate the structure, referencing entities by `@id`.

In [ ]:
# List all record sets in the dataset along with their fields, using `@id`
record_sets = list(dataset.record_sets.values())

if not record_sets:
    print("No record sets found in the dataset metadata. Attempting to enumerate logical tables...")
    for logical_table in dataset.logical_tables.values():
        print(f"LogicalTable @id: {logical_table.id}, name: {getattr(logical_table, 'name', None)}")
        if hasattr(logical_table, 'fields'):
            for field in logical_table.fields:
                print(f"  Field @id: {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set.id}, name: {getattr(record_set, 'name', None)}")
        for field in record_set.fields:
            print(f"  Field @id: {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")

# For reproducibility, gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets.values()]
print("\nAll available record sets (by @id):")
for rid in record_set_ids:
    print(f" - {rid}")


## 3. Data Extraction

We load one or more record sets (tables) into pandas DataFrames, referencing each by its Croissant schema `@id`.

In [ ]:
# Find all record set @ids (using fallback if necessary)
if record_set_ids:
    used_record_set_ids = record_set_ids
else:
    # Fallback: sometimes Croissant datasets have logical_tables but not explicit record sets
    used_record_set_ids = list(dataset.logical_tables.keys())

# Load each record set by @id into a DataFrame
dataframes = {}
for record_set_id in used_record_set_ids:
    print(f"Loading record set @id: {record_set_id}")
    # Use the records() method referencing by @id
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" => Loaded {len(df)} records with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"[Warning] Failed to load records for {record_set_id}: {e}")

# Display the first few rows of the primary table (selected by the first @id)
if used_record_set_ids:
    main_record_set_id = used_record_set_ids[0]
    print(f"\nFirst few rows for '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available for data extraction.")


## 4. Exploratory Data Analysis (EDA)

Let's examine numeric fields in the main record set. We'll:
- Select a numeric field by its `@id` (if available)
- Filter rows where the field is above a threshold
- Normalize this field
- Group by a categorical field (if applicable)

> **Note:** You may review the field data types in the overview cell above to adapt the following to your needs.

In [ ]:
main_df = dataframes.get(main_record_set_id)
if main_df is not None and not main_df.empty:
    # Infer numeric field(s) by pandas dtype
    numeric_candidates = [c for c in main_df.columns if pd.api.types.is_numeric_dtype(main_df[c]) and not main_df[c].isnull().all()]
    print(f"Numeric fields in main DataFrame: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # e.g., 'age' or other suitable field
        print(f"Using numeric field: {numeric_field}")
        # Filter records
        threshold = main_df[numeric_field].mean()
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field}:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by the first categorical (object) field
        group_candidates = [c for c in main_df.columns if main_df[c].dtype=='object' and main_df[c].nunique() < 20]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by categorical field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric fields detected in the main table.")
else:
    print("No data loaded for EDA.")


## 5. Visualization

Let's visualize some aspects of the main record set. We'll plot the distribution of a numeric field and, if available, compare means across a categorical field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and not main_df.empty and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Visualization skipped: numeric field not available.")

## 6. Conclusion

In this notebook, we used `mlcroissant` to:
- Load and inspect the FAIR^2 dataset metadata (leveraging Croissant `@id` fields for referencing)
- List all available record sets and their fields
- Extract record sets into pandas DataFrames for examination
- Perform basic EDA, filtering, normalizing, and grouping summary statistics
- Visualize data as a histogram and, where feasible, as categorical boxplots

Explore further by referencing Croissant `@id`s as needed to link dataset schema, fields, and downstream analyses.
